In [2]:
import os
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import seaborn as sns

from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
from sklearn.model_selection import train_test_split

### Reusable Function

In [3]:
def class_counts(label_array, label_cols):
    counts = label_array.sum(axis=0)
    return pd.Series(counts, index=label_cols)

def class_percentages(label_array):
    counts = label_array.sum(axis=0)
    total = label_array.shape[0]
    return pd.Series((counts / total) * 100, index=label_cols)

In [4]:
def merge_case_insensitive_columns(df):
    # Map lowercase column names to their original variants
    col_map = {}
    for col in df.columns:
        key = col.lower()
        col_map.setdefault(key, []).append(col)

    # Build merged columns
    merged_cols = []
    for key, variants in col_map.items():
        if len(variants) == 1:
            col = df[variants[0]].copy()
            col.name = variants[0]
        else:
            col = df[variants].max(axis=1)
            col.name = key.title()
        merged_cols.append(col)

    # Concatenate all columns at once to avoid fragmentation
    merged_df = pd.concat(merged_cols, axis=1)

    return merged_df

In [5]:
def safe_split(x):
    if isinstance(x, str):
        return [item.strip() for item in x.split(';') if item.strip()]
    elif isinstance(x, list):
        return [str(item).strip() for item in x if str(item).strip()]
    else:
        return []

## NIH Dataset Splitting

In [14]:
csv_dir = r"nih_chestxray14_Data_Entry_2017.csv"
nih_sheet = pd.read_csv(csv_dir)

# Filter the labels as several columns after symbol "|"
nih_split_sheet = nih_sheet['Finding Labels'].str.split('|', expand=True)
nih_split_sheet.columns = [f'tag_{i+1}' for i in range(nih_split_sheet.shape[1])]
nih_sheet = nih_sheet.join(nih_split_sheet)
column_kept = ['Image Index', 'tag_1', 'tag_2', 'tag_3', 'tag_4', 'tag_5', 'tag_6', 'tag_7', 'tag_8', 'tag_9']
nih_sheet_tags = nih_sheet[column_kept]

# Get all unique values for list of possible pathologies
unique_values = nih_sheet_tags['tag_1'].unique().tolist()

columns_to_check = ['tag_2', 'tag_3', 'tag_4', 'tag_5', 'tag_6', 'tag_7', 'tag_8', 'tag_9']

for col in columns_to_check:
    for value in nih_sheet_tags[col]:
        if value not in unique_values:
            unique_values.append(value)

print(unique_values)

['Cardiomegaly', 'No Finding', 'Hernia', 'Mass', 'Infiltration', 'Effusion', 'Nodule', 'Emphysema', 'Atelectasis', 'Pleural_Thickening', 'Pneumothorax', 'Fibrosis', 'Consolidation', 'Edema', 'Pneumonia', None]


In [15]:
# Making the final binary matrix for pathologies
nih_final_sheet = nih_sheet.copy()

for col in unique_values:
    nih_final_sheet.loc[:, col] = nih_final_sheet['Finding Labels'].apply(
        lambda x: 1 if col in str(x).split('|') else 0
    )

# Filter out unnecessary columns for binary matrix
column_kept = ['Image Index', 'Patient ID', 'No Finding', 'Cardiomegaly', 'Hernia', 'Mass', 'Infiltration', 'Effusion', 'Nodule', 'Emphysema', 'Atelectasis', 'Pleural_Thickening', 'Pneumothorax', 'Fibrosis', 'Consolidation', 'Edema', 'Pneumonia']
nih_final_sheet = nih_final_sheet[column_kept]

In [16]:
# Create label DataFrame & Print summary statistics
label_cols = ['No Finding', 'Cardiomegaly', 'Hernia', 'Mass', 'Infiltration', 'Effusion', 'Nodule', 'Emphysema', 'Atelectasis', 'Pleural_Thickening', 'Pneumothorax', 'Fibrosis', 'Consolidation', 'Edema', 'Pneumonia']
labels_nih = nih_final_sheet[label_cols].fillna(0).astype(int)

print("Samples:", len(nih_final_sheet))
print(labels_nih.sum().sort_values(ascending=False))

Samples: 112120
No Finding            60361
Infiltration          19894
Effusion              13317
Atelectasis           11559
Nodule                 6331
Mass                   5782
Pneumothorax           5302
Consolidation          4667
Pleural_Thickening     3385
Cardiomegaly           2776
Emphysema              2516
Edema                  2303
Fibrosis               1686
Pneumonia              1431
Hernia                  227
dtype: int64


In [17]:
# Split dataset into train/val/test with patient-wise splitting
random_state = 6033689
test_ratio = 0.2
val_ratio_of_holdout = 0.5

patient_col = "Patient ID"
label_cols = list(labels_nih.columns) if hasattr(labels_nih, "columns") else list(range(y.shape[1]))

# Create patient Specific Dataframe
df = nih_final_sheet.copy()
df[label_cols] = df[label_cols].astype(int)
patient_grouped = df.groupby(patient_col)[label_cols].max()
patient_ids = patient_grouped.index.to_numpy()
patient_labels = patient_grouped.values.astype(int) 

# First split patients (80/20) to train:temp (selection & test)
msss = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=test_ratio, random_state=random_state)
train_pat_idx, temp_pat_idx = next(msss.split(patient_ids, patient_labels))

train_patient_ids = patient_ids[train_pat_idx]
temp_patient_ids = patient_ids[temp_pat_idx]
temp_patient_labels = patient_labels[temp_pat_idx]

# Second split patients (50/50) to selection & test
msss2 = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=val_ratio_of_holdout, random_state=random_state)
selection_rel_idx, test_rel_idx = next(msss2.split(temp_patient_ids, temp_patient_labels))

selection_patient_ids = temp_patient_ids[selection_rel_idx]
test_patient_ids  = temp_patient_ids[test_rel_idx]

train_df = nih_final_sheet[nih_final_sheet[patient_col].isin(train_patient_ids)].copy()
selection_df = nih_final_sheet[nih_final_sheet[patient_col].isin(selection_patient_ids)].copy()
test_df = nih_final_sheet[nih_final_sheet[patient_col].isin(test_patient_ids)].copy()

print("Patient counts -> total/patients:", len(patient_ids))
print("Sizes -> train/val/test (patients):", len(train_patient_ids), len(selection_patient_ids), len(test_patient_ids))
print("Sizes -> train/val/test (rows):", len(train_df), len(selection_df), len(test_df))

Patient counts -> total/patients: 30805
Sizes -> train/val/test (patients): 24644 3102 3059
Sizes -> train/val/test (rows): 89905 11154 11061


In [18]:
# Making sure no patient shown twice
assert set(train_patient_ids).isdisjoint(set(test_patient_ids)), "Train/Test overlap!"
assert set(train_patient_ids).isdisjoint(set(selection_patient_ids)), "Train/Selection overlap!"
assert set(test_patient_ids).isdisjoint(set(selection_patient_ids)), "Test/Selection overlap!"

print("No patient ID overlap across train/val/test splits")

No patient ID overlap across train/val/test splits


In [19]:
full_pct = class_percentages(labels_nih.values)
train_pct = class_percentages(train_df[label_cols].values)
selection_pct = class_percentages(selection_df[label_cols].values)
test_pct = class_percentages(test_df[label_cols].values)

report_pct = pd.DataFrame({
    "full_pct": full_pct,
    "train_pct": train_pct,
    "selection_pct": selection_pct,
    "test_pct": test_pct,
})

report_pct = report_pct.sort_values("full_pct", ascending=False)
report_pct

,full_pct,train_pct,selection_pct,test_pct
No Finding,53.836068,53.836828,54.016496,53.647952
Infiltration,17.743489,17.732050,17.625964,17.954977
Effusion,11.877453,11.915911,11.663977,11.780128
Atelectasis,10.309490,10.320894,10.319168,10.207034
Nodule,5.646629,5.558089,5.621302,6.391827
Mass,5.156975,5.175463,4.877174,5.288853
Pneumothorax,4.728862,4.831767,4.276493,4.348612
Consolidation,4.162504,4.162171,4.357181,3.968900
Pleural_Thickening,3.019087,3.047661,2.797203,3.010578
Cardiomegaly,2.475919,2.430343,2.653756,2.667028


In [20]:
full_count = class_counts(labels_nih.values, label_cols)
train_count = class_counts(train_df[label_cols].values, label_cols)
selection_count = class_counts(selection_df[label_cols].values, label_cols)
test_count = class_counts(test_df[label_cols].values, label_cols)

report_count = pd.DataFrame({
    "full_count": full_count,
    "train_count": train_count,
    "selection_count": selection_count,
    "test_count": test_count,
})

report_count = report_count.sort_values("full_count", ascending=False)
report_count

,full_count,train_count,selection_count,test_count
No Finding,60361,48402,6025,5934
Infiltration,19894,15942,1966,1986
Effusion,13317,10713,1301,1303
Atelectasis,11559,9279,1151,1129
Nodule,6331,4997,627,707
Mass,5782,4653,544,585
Pneumothorax,5302,4344,477,481
Consolidation,4667,3742,486,439
Pleural_Thickening,3385,2740,312,333
Cardiomegaly,2776,2185,296,295


In [21]:
# Saving the split in CSV for TorchXRayVision data loader
train_df.to_csv("nih_train_split.csv", index=False)
selection_df.to_csv("nih_selection_split.csv", index=False)
test_df.to_csv("nih_test_split.csv", index=False)

## Open-I Dataset Splitting

In [26]:
# Load the data label directory files
csv1_dir = r"indiana_projections.csv"
csv2_dir = r"indiana_reports.csv"
projection_sheet = pd.read_csv(csv1_dir)
reports_sheet = pd.read_csv(csv2_dir)

# Merge the two sheets with UID
merged_sheet = pd.merge(projection_sheet, reports_sheet, on='uid', how='inner')
column_kept = ['filename', 'Problems']
filtered_simple_sheet = merged_sheet[column_kept]

In [27]:
# Split the sheet into binary columns for each pathology (in Problems column)
filtered_simple_sheet = filtered_simple_sheet.copy()
filtered_simple_sheet['Problems'] = filtered_simple_sheet['Problems'].apply(safe_split)
expanded_sheet = filtered_simple_sheet['Problems'].apply(lambda x: pd.Series(1, index=set(x))).fillna(0)
filtered_simple_sheet = filtered_simple_sheet.drop(columns='Problems').join(expanded_sheet)
filtered_simple_sheet = merge_case_insensitive_columns(filtered_simple_sheet)

In [28]:
# Group similar pathologies into one column (from Ollama LLM grouping)
filtered_simple_sheet["Pneumothorax"] = filtered_simple_sheet[["Pneumothorax", "Subcutaneous Emphysema", "Hydropneumothorax", "Hemopneumothorax"]].max(axis=1)
filtered_simple_sheet["Pulmonary Atelectasis"] = filtered_simple_sheet[["Pulmonary Atelectasis", "Volume Loss"]].max(axis=1)
filtered_simple_sheet["Pulmonary Edema"] = filtered_simple_sheet[["Pulmonary Edema", "Pulmonary Congestion"]].max(axis=1)
filtered_simple_sheet["Opacity"] = filtered_simple_sheet[["Opacity", "Airspace Disease", "Density", "Infiltrate", "Pulmonary Fibrosis", "Fibrosis"]].max(axis=1)
filtered_simple_sheet["Mass"] = filtered_simple_sheet[["Mass", "Cicatrix", "Cavitation", "Granuloma", "Calcified Granuloma", "Granulomatous Disease", "Nodule", "Cysts", "Emphysema", "Bullous Emphysema", "Pulmonary Emphysema", "Sarcoidosis"]].max(axis=1)
filtered_simple_sheet["Thickening"] = filtered_simple_sheet[["Thickening", "Diaphragmatic Eventration"]].max(axis=1)
filtered_simple_sheet["Pleural Effusion"] = filtered_simple_sheet[["Pleural Effusion", "Hemothorax"]].max(axis=1)
filtered_simple_sheet["Medical Device"] = filtered_simple_sheet[["Medical Device", "Catheters, Indwelling", "Tube, Inserted", "Stents", "Implanted Medical Device"]].max(axis=1)
filtered_simple_sheet["Aorta"] = filtered_simple_sheet[["Aorta", "Heart Failure", "Cardiac Shadow", "Aorta, Thoracic", "Aortic Aneurysm"]].max(axis=1)
filtered_simple_sheet["Hernia, Diaphragmatic"] = filtered_simple_sheet[["Hernia, Diaphragmatic", "Hernia, Hiatal"]].max(axis=1)

# Drop combined columns
filtered_simple_sheet = filtered_simple_sheet.drop(columns=["Subcutaneous Emphysema", "Hydropneumothorax", "Hemopneumothorax"])
filtered_simple_sheet = filtered_simple_sheet.drop(columns=["Volume Loss"])
filtered_simple_sheet = filtered_simple_sheet.drop(columns=["Pulmonary Congestion"])
filtered_simple_sheet = filtered_simple_sheet.drop(columns=["Airspace Disease", "Density", "Infiltrate", "Pulmonary Fibrosis", "Fibrosis"])
filtered_simple_sheet = filtered_simple_sheet.drop(columns=["Cicatrix", "Cavitation", "Granuloma", "Calcified Granuloma", "Granulomatous Disease", "Nodule", "Cysts", "Emphysema", "Bullous Emphysema", "Pulmonary Emphysema", "Sarcoidosis"])
filtered_simple_sheet = filtered_simple_sheet.drop(columns=["Hemothorax"])
filtered_simple_sheet = filtered_simple_sheet.drop(columns=["Catheters, Indwelling", "Tube, Inserted", "Stents", "Implanted Medical Device"])
filtered_simple_sheet = filtered_simple_sheet.drop(columns=["Heart Failure", "Cardiac Shadow", "Aorta, Thoracic", "Aortic Aneurysm"])
filtered_simple_sheet = filtered_simple_sheet.drop(columns=["Hernia, Hiatal"])

In [29]:
# Eliminate any columns that's not selected in final label columns
label_cols = ['normal', 'Pneumonia', 'Pneumothorax', 'Pulmonary Atelectasis', 'Pulmonary Edema', 'Cardiomegaly', 'Consolidation', 'Opacity',
             'Mass', 'Pleural Effusion', 'Medical Device', 'Fractures, Bone', 'Aorta']
labels_openi = filtered_simple_sheet[label_cols].fillna(0).astype(int)

# Check total labels from dataset
print("Samples:", len(filtered_simple_sheet))
print(labels_openi.sum().sort_values(ascending=False))

Samples: 7466
normal                   2695
Mass                     1383
Opacity                  1225
Cardiomegaly              655
Pulmonary Atelectasis     634
Aorta                     580
Medical Device            425
Pleural Effusion          286
Pulmonary Edema           183
Fractures, Bone           163
Pneumonia                  77
Pneumothorax               56
Consolidation              53
dtype: int64


In [30]:
# Split dataset into train/selection/test with patient-wise splitting
random_state = 6033689
train_test_ratio = 0.2  #(80/20, then 50/50)

X = filtered_simple_sheet.index.values
y = labels_openi.values

# First split: 80/20
msss = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=train_test_ratio, random_state=random_state)
train_idx, temp_idx = next(msss.split(X, y))

temp_X = X[temp_idx]
temp_y = y[temp_idx]

# Second split: 50/50
msss2 = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.5, random_state=random_state)
test_rel_idx, val_rel_idx = next(msss2.split(temp_X, temp_y))

selection_idx = temp_idx[test_rel_idx]
test_idx = temp_idx[val_rel_idx]

# Create DataFrames for each set
train_df = filtered_simple_sheet.loc[train_idx].copy()
selection_df = filtered_simple_sheet.loc[selection_idx].copy()
test_df = filtered_simple_sheet.loc[test_idx].copy()

print("Sizes -> train/selection/test:", len(train_df), len(selection_df), len(test_df))

Sizes -> train/selection/test: 5972 747 747


In [31]:
full_pct = class_percentages(labels_openi.values)
train_pct = class_percentages(labels_openi.loc[train_idx].values)
selection_pct = class_percentages(labels_openi.loc[selection_idx].values)
test_pct = class_percentages(labels_openi.loc[test_idx].values)

report_pct = pd.DataFrame({
    "full_pct": full_pct,
    "train_pct": train_pct,
    "selection_pct": selection_pct,
    "test_pct": test_pct,
})

report_pct = report_pct.sort_values("full_pct", ascending=False)
report_pct

,full_pct,train_pct,selection_pct,test_pct
normal,36.096973,36.101808,36.144578,36.010710
Mass,18.523975,18.519759,18.607764,18.473896
Opacity,16.407715,16.409913,16.331995,16.465863
Cardiomegaly,8.773105,8.774280,8.835341,8.701473
Pulmonary Atelectasis,8.491830,8.489618,8.433735,8.567604
Aorta,7.768551,7.769591,7.764391,7.764391
Medical Device,5.692473,5.693235,5.756359,5.622490
Pleural Effusion,3.830699,3.834561,3.882195,3.748327
Pulmonary Edema,2.451112,2.444742,2.543507,2.409639
"Fractures, Bone",2.183231,2.176825,2.141901,2.275770


In [32]:
full_count = class_counts(labels_openi.values, label_cols)
train_count = class_counts(labels_openi.loc[train_idx].values, label_cols)
selection_count = class_counts(labels_openi.loc[selection_idx].values, label_cols)
test_count = class_counts(labels_openi.loc[test_idx].values, label_cols)

report_count = pd.DataFrame({
    "full_count": full_count,
    "train_count": train_count,
    "selection_count": selection_count,
    "test_count": test_count,
})

report_count = report_count.sort_values("full_count", ascending=False)
report_count

,full_count,train_count,selection_count,test_count
normal,2695,2156,270,269
Mass,1383,1106,139,138
Opacity,1225,980,122,123
Cardiomegaly,655,524,66,65
Pulmonary Atelectasis,634,507,63,64
Aorta,580,464,58,58
Medical Device,425,340,43,42
Pleural Effusion,286,229,29,28
Pulmonary Edema,183,146,19,18
"Fractures, Bone",163,130,16,17


In [33]:
# Saving the split in CSV for TorchXRayVision data loader
train_df.to_csv("openi_train_split.csv", index=False)
selection_df.to_csv("openi_selection_split.csv", index=False)
test_df.to_csv("openi_test_split.csv", index=False)

## Padchest Dataset Splitting

In [36]:
# Load the data label directory files
csv_dir = r"padchest_chest_x_ray_images_labels_160K_01.02.19.csv"
image_sheet = pd.read_csv(csv_dir)
column_kept = ['ImageID', 'ImageDir', 'PatientID', 'Projection', 'Labels']
filter_sheet = image_sheet[column_kept]

In [37]:
# Break down all the labels with one-hot Encoding
import ast

filter_sheet = filter_sheet.copy()
filter_sheet['Labels'] = filter_sheet['Labels'].apply(lambda x: ast.literal_eval(x) if pd.notnull(x) else [])
expanded_sheet = filter_sheet['Labels'].apply(lambda x: pd.Series(1, index=x)).fillna(0)
padchest_final_sheet = filter_sheet.drop(columns='Labels').join(expanded_sheet)
padchest_final_sheet = padchest_final_sheet.drop(['ImageDir', 'Projection'], axis=1)

In [38]:
# Check duplicates causes by space -- eg. "fracture" and " fracture"
padchest_final_sheet.columns = padchest_final_sheet.columns.str.strip()
original_cols = padchest_final_sheet.columns.tolist()
raw_cols = padchest_final_sheet.columns.str.strip().tolist()

# Find duplicates caused by spacing
from collections import Counter
col_counts = Counter(raw_cols)

# Merge columns with same stripped name
for col in col_counts:
    if col_counts[col] > 1:
        # Find all variants of this column name
        variants = [c for c in original_cols if c.strip() == col]
        # Merge them: 1 if any variant is 1, else 0
        padchest_final_sheet[col] = padchest_final_sheet[variants].fillna(0).astype(int).max(axis=1)
        # Drop all but one
        to_drop = [c for c in variants if c != col]
        padchest_final_sheet.drop(columns=to_drop, inplace=True)

# Merge column with insensitive same columns name
padchest_final_sheet = merge_case_insensitive_columns(padchest_final_sheet)

In [39]:
# Group similar pathologies into one column (from Ollama LLM grouping)
padchest_final_sheet["Pneumonia"] = padchest_final_sheet[["Pneumonia", "Atypical Pneumonia"]].max(axis=1)
padchest_final_sheet["Pneumothorax"] = padchest_final_sheet[["Pneumothorax", "air fluid level", "Hydropneumothorax"]].max(axis=1)
padchest_final_sheet["Atelectasis"] = padchest_final_sheet[["Atelectasis", "Laminar Atelectasis", "Segmental Atelectasis", "Lobar Atelectasis", "atelectasis basal", "Total Atelectasis", "Round Atelectasis", "Flattened Diaphragm", "Volume Loss", "hypoexpansion basal", "Hypoexpansion"]].max(axis=1)
padchest_final_sheet["Cardiomegaly"] = padchest_final_sheet[["Cardiomegaly", "Heart Insufficiency"]].max(axis=1)
padchest_final_sheet["Pulmonary Edema"] = padchest_final_sheet[["Pulmonary Edema", "Kerley Lines"]].max(axis=1)
padchest_final_sheet["Infiltrates"] = padchest_final_sheet[["Infiltrates", "Alveolar Pattern", "Interstitial Pattern", "Ground Glass Pattern", "Reticulonodular Interstitial Pattern", "Reticular Interstitial Pattern", "Bronchovascular Markings", "Miliary Opacities"]].max(axis=1)
padchest_final_sheet["Pulmonary Mass"] = padchest_final_sheet[["Pulmonary Mass", "Mass", "Soft Tissue Mass", "pleural mass", "Nodule", "Multiple Nodules", "Cavitation", "Calcified Adenopathy", "Lymphangitis Carcinomatosa", "Lepidic Adenocarcinoma", "Bone Metastasis", "Lung Metastasis", "Tuberculosis Sequelae", "Pulmonary Fibrosis", "Calcified Granuloma", "Granuloma", "cyst"]].max(axis=1)
padchest_final_sheet["Pleural Thickening"] = padchest_final_sheet[["Pleural Thickening", "Calcified Pleural Thickening", "Apical Pleural Thickening", "Major Fissure Thickening", "Minor Fissure Thickening", "Fissure Thickening", "Pleural Plaques", "Calcified Pleural Plaques", "Diaphragmatic Eventration"]].max(axis=1)
padchest_final_sheet["Pleural Effusion"] = padchest_final_sheet[["Pleural Effusion", "Loculated Fissural Effusion"]].max(axis=1)
padchest_final_sheet["Endotracheal Tube"] = padchest_final_sheet[["Endotracheal Tube", "Tracheostomy Tube", "NSG tube", "Single Chamber Device", "Dual Chamber Device", "Electrical Device", "Central Venous Catheter Via Subclavian Vein", "Central Venous Catheter", "Reservoir Central Venous Catheter", "Central Venous Catheter Via Jugular Vein", "Central Venous Catheter Via Umbilical Vein", "Catheter", "double J stent", "Chest Drain Tube", "Gastrostomy Tube", "Ventriculoperitoneal Drain Tube", "Artificial Heart Valve", "Artificial Mitral Heart Valve", "Artificial Aortic Heart Valve", "Pacemaker", "bone cement", "endoprosthesis", "Aortic Endoprosthesis"]].max(axis=1)
padchest_final_sheet["fracture"] = padchest_final_sheet[["fracture", "Clavicle Fracture", "Callus Rib Fracture", "Humeral Fracture", "Vertebral Fracture", "Rib Fracture"]].max(axis=1)
padchest_final_sheet["Aortic Elongation"] = padchest_final_sheet[["Aortic Elongation", "Supra Aortic Elongation", "Descendent Aortic Elongation", "Ascendent Aortic Elongation", "Mediastinal Enlargement", "Pulmonary Artery Enlargement", "Aortic Button Enlargement", "Superior Mediastinal Enlargement", "Mediastinal Shift"]].max(axis=1)
padchest_final_sheet["abnormal foreign body"] = padchest_final_sheet[["abnormal foreign body", "External Foreign Body", "Metal", "Suture Material", "Osteosynthesis Material"]].max(axis=1)

# Drop combined columns
padchest_final_sheet = padchest_final_sheet.drop(columns=["Atypical Pneumonia"])
padchest_final_sheet = padchest_final_sheet.drop(columns=["air fluid level", "Hydropneumothorax"])
padchest_final_sheet = padchest_final_sheet.drop(columns=["Laminar Atelectasis", "Segmental Atelectasis", "Lobar Atelectasis", "atelectasis basal", "Total Atelectasis", "Round Atelectasis", "Flattened Diaphragm", "Volume Loss", "hypoexpansion basal", "Hypoexpansion"])
padchest_final_sheet = padchest_final_sheet.drop(columns=["Heart Insufficiency"])
padchest_final_sheet = padchest_final_sheet.drop(columns=["Kerley Lines"])
padchest_final_sheet = padchest_final_sheet.drop(columns=["Alveolar Pattern", "Interstitial Pattern", "Ground Glass Pattern", "Reticulonodular Interstitial Pattern", "Reticular Interstitial Pattern", "Bronchovascular Markings", "Miliary Opacities"])
padchest_final_sheet = padchest_final_sheet.drop(columns=["Mass", "Soft Tissue Mass", "pleural mass", "Nodule", "Multiple Nodules", "Cavitation", "Calcified Adenopathy", "Lymphangitis Carcinomatosa", "Lepidic Adenocarcinoma", "Bone Metastasis", "Lung Metastasis", "Tuberculosis Sequelae", "Pulmonary Fibrosis", "Calcified Granuloma", "Granuloma", "cyst"])
padchest_final_sheet = padchest_final_sheet.drop(columns=["Calcified Pleural Thickening", "Apical Pleural Thickening", "Major Fissure Thickening", "Minor Fissure Thickening", "Fissure Thickening", "Pleural Plaques", "Calcified Pleural Plaques", "Diaphragmatic Eventration"])
padchest_final_sheet = padchest_final_sheet.drop(columns=["Loculated Fissural Effusion"])
padchest_final_sheet = padchest_final_sheet.drop(columns=["Tracheostomy Tube", "NSG tube", "Single Chamber Device", "Dual Chamber Device", "Electrical Device", "Central Venous Catheter Via Subclavian Vein", "Central Venous Catheter", "Reservoir Central Venous Catheter", "Central Venous Catheter Via Jugular Vein", "Central Venous Catheter Via Umbilical Vein", "Catheter", "double J stent", "Chest Drain Tube", "Gastrostomy Tube", "Ventriculoperitoneal Drain Tube", "Artificial Heart Valve", "Artificial Mitral Heart Valve", "Artificial Aortic Heart Valve", "Pacemaker", "bone cement", "endoprosthesis", "Aortic Endoprosthesis"])
padchest_final_sheet = padchest_final_sheet.drop(columns=["Clavicle Fracture", "Callus Rib Fracture", "Humeral Fracture", "Vertebral Fracture", "Rib Fracture"])
padchest_final_sheet = padchest_final_sheet.drop(columns=["Supra Aortic Elongation", "Descendent Aortic Elongation", "Ascendent Aortic Elongation", "Mediastinal Enlargement", "Pulmonary Artery Enlargement", "Aortic Button Enlargement", "Superior Mediastinal Enlargement", "Mediastinal Shift"])
padchest_final_sheet = padchest_final_sheet.drop(columns=["External Foreign Body", "Metal", "Suture Material", "Osteosynthesis Material"])

In [43]:
# Filter out unnecessary columns for binary matrix
label_cols = ['ImageID', 'PatientID', 'Normal', 'Pneumonia', 'Pneumothorax', 'Atelectasis', 'Cardiomegaly', 'Pulmonary Edema', 'Consolidation', 'Infiltrates', 'Pulmonary Mass', 'Pleural Effusion', 'Endotracheal Tube', 'fracture', 'Aortic Elongation']
labels_padchest = padchest_final_sheet[label_cols]

print("Samples:", len(padchest_final_sheet))

Samples: 160861


In [44]:
# Split dataset into train/val/test with patient-wise splitting
random_state = 6033689
test_ratio = 0.2
val_ratio_of_holdout = 0.5

patient_col = "PatientID"
label_cols = labels_padchest.select_dtypes(include="number").columns.tolist()

# Create Patient Specific Dataframe
df = labels_padchest.copy()
df[label_cols] = df[label_cols].astype(int)
patient_grouped = df.groupby(patient_col)[label_cols].max()
patient_ids = patient_grouped.index.to_numpy()
patient_labels = patient_grouped.values.astype(int) 

# First split patients (80/20) to train:temp (selection & test)
msss = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=test_ratio, random_state=random_state)
train_pat_idx, temp_pat_idx = next(msss.split(patient_ids, patient_labels))

train_patient_ids = patient_ids[train_pat_idx]
temp_patient_ids = patient_ids[temp_pat_idx]
temp_patient_labels = patient_labels[temp_pat_idx]

# Second split patients (50/50) to selection & test
msss2 = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=val_ratio_of_holdout, random_state=random_state)
selection_rel_idx, test_rel_idx = next(msss2.split(temp_patient_ids, temp_patient_labels))

selection_patient_ids = temp_patient_ids[selection_rel_idx]
test_patient_ids  = temp_patient_ids[test_rel_idx]

train_df = labels_padchest[labels_padchest[patient_col].isin(train_patient_ids)].copy()
selection_df = labels_padchest[labels_padchest[patient_col].isin(selection_patient_ids)].copy()
test_df = labels_padchest[labels_padchest[patient_col].isin(test_patient_ids)].copy()

print("Patient counts -> total/patients:", len(patient_ids))
print("Sizes -> train/val/test (patients):", len(train_patient_ids), len(selection_patient_ids), len(test_patient_ids))
print("Sizes -> train/val/test (rows):", len(train_df), len(selection_df), len(test_df))

Patient counts -> total/patients: 67625
Sizes -> train/val/test (patients): 54100 6762 6763
Sizes -> train/val/test (rows): 128534 16038 16289


In [45]:
labels_only = labels_padchest.drop(columns=["ImageID", "PatientID"])

full_pct = class_percentages(labels_only.values)
train_pct = class_percentages(train_df[label_cols].values)
selection_pct = class_percentages(selection_df[label_cols].values)
test_pct = class_percentages(test_df[label_cols].values)

report_pct = pd.DataFrame({
    "full_pct": full_pct,
    "train_pct": train_pct,
    "selection_pct": selection_pct,
    "test_pct": test_pct,
})

report_pct = report_pct.sort_values("full_pct", ascending=False)
report_pct

,full_pct,train_pct,selection_pct,test_pct
Normal,31.465675,31.527067,31.288191,31.155995
Infiltrates,12.367821,12.381938,12.021449,12.597458
Cardiomegaly,10.198246,10.170850,10.063599,10.546995
Endotracheal Tube,8.818172,8.936935,8.330216,8.361471
Atelectasis,8.389852,8.361212,8.610799,8.398306
Aortic Elongation,8.115702,8.109916,8.018456,8.257106
Pulmonary Mass,7.326201,7.313240,7.382467,7.373074
Pleural Effusion,6.170545,6.101887,6.154134,6.728467
Pneumonia,5.081406,5.103708,4.900860,5.083185
fracture,2.878883,2.895732,2.911834,2.713488


In [46]:
full_count = class_counts(labels_only.values, label_cols)
train_count = class_counts(train_df[label_cols].values, label_cols)
selection_count = class_counts(selection_df[label_cols].values, label_cols)
test_count = class_counts(test_df[label_cols].values, label_cols)

report_count = pd.DataFrame({
    "full_count": full_count,
    "train_count": train_count,
    "selection_count": selection_count,
    "test_count": test_count,
})

report_count = report_count.sort_values("full_count", ascending=False)
report_count

,full_count,train_count,selection_count,test_count
Normal,50616.0,40523.0,5018.0,5075.0
Infiltrates,19895.0,15915.0,1928.0,2052.0
Cardiomegaly,16405.0,13073.0,1614.0,1718.0
Endotracheal Tube,14185.0,11487.0,1336.0,1362.0
Atelectasis,13496.0,10747.0,1381.0,1368.0
Aortic Elongation,13055.0,10424.0,1286.0,1345.0
Pulmonary Mass,11785.0,9400.0,1184.0,1201.0
Pleural Effusion,9926.0,7843.0,987.0,1096.0
Pneumonia,8174.0,6560.0,786.0,828.0
fracture,4631.0,3722.0,467.0,442.0


In [47]:
# Saving the split in CSV for TorchXRayVision data loader
train_df.to_csv("padchest_train_split.csv", index=False)
selection_df.to_csv("padchest_selection_split.csv", index=False)
test_df.to_csv("padchest_test_split.csv", index=False)

## RSNA Dataset

In [28]:
rsna = pd.read_csv(r"/home/varrel/imaging/all_images/RSNA_dataset/stage_2_train_labels.csv")
rsna = rsna.drop_duplicates(subset = ['patientId'])

In [29]:
# Split dataset into train/selection/test (no patient-wise splitting info available)
X = rsna['patientId']
y = rsna['Target']

# First split: train (80%)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

# Second split: selection (10%) and test (10%)
X_selection, X_test, y_selection, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.5,
    stratify=y_temp,
    random_state=42
)

train_df = rsna.loc[X_train.index]
selection_df = rsna.loc[X_selection.index]
test_df = rsna.loc[X_test.index]

In [30]:
# Print sizes and distributions
print("Sizes -> train/selection/test:", len(train_df), len(selection_df), len(test_df))
print("Train distribution:\n", train_df['Target'].value_counts(normalize=True))
print("Selection distribution:\n", selection_df['Target'].value_counts(normalize=True))
print("Test distribution:\n", test_df['Target'].value_counts(normalize=True))

Sizes -> train/selection/test: 21347 2668 2669
Train distribution:
 Target
0    0.774676
1    0.225324
Name: proportion, dtype: float64
Selection distribution:
 Target
0    0.774738
1    0.225262
Name: proportion, dtype: float64
Test distribution:
 Target
0    0.774822
1    0.225178
Name: proportion, dtype: float64


In [53]:
# Saving the split in CSV for TorchXRayVision data loader
train_df.to_csv("rsna_train_split.csv", index=False)
selection_df.to_csv("rsna_selection_split.csv", index=False)
test_df.to_csv("rsna_test_split.csv", index=False)

## SIIM-ACR

In [7]:
siim = pd.read_csv(r"/home/varrel/imaging/all_images/SIIM_dataset/train-rle.csv")
siim['EncodedPixels'] = np.where(siim[' EncodedPixels'] == "-1", -1, 0)

In [25]:
# Split dataset into train/selection/test (no patient-wise splitting info available)
X = siim['ImageId']
y = siim['EncodedPixels']

# First split: train (80%)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

# Second split: selection (10%) and test (10%)
X_selection, X_test, y_selection, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.5,
    stratify=y_temp,
    random_state=42
)

train_df = siim.loc[X_train.index]
selection_df = siim.loc[X_selection.index]
test_df = siim.loc[X_test.index]

In [26]:
# Print sizes and distributions
print("Sizes -> train/selection/test:", len(train_df), len(selection_df), len(test_df))
print("Train distribution:\n", train_df['EncodedPixels'].value_counts(normalize=True))
print("Selection distribution:\n", selection_df['EncodedPixels'].value_counts(normalize=True))
print("Test distribution:\n", test_df['EncodedPixels'].value_counts(normalize=True))

Sizes -> train/selection/test: 10363 1295 1296
Train distribution:
 EncodedPixels
-1    0.723922
 0    0.276078
Name: proportion, dtype: float64
Selection distribution:
 EncodedPixels
-1    0.724324
 0    0.275676
Name: proportion, dtype: float64
Test distribution:
 EncodedPixels
-1    0.723765
 0    0.276235
Name: proportion, dtype: float64


In [10]:
# Saving the split in CSV for TorchXRayVision data loader
train_df.to_csv("siim_train_split.csv", index=False)
selection_df.to_csv("siim_selection_split.csv", index=False)
test_df.to_csv("siim_test_split.csv", index=False)

## CheXpert

In [12]:
chex = pd.read_csv(r"/home/varrel/imaging/all_images/CheX_dataset/train_visualCheXbert.csv")
chex["patientID"] = chex["Path"].str.split("/").str[2]

In [13]:
# Create label DataFrame & Print summary statistics
label_cols = ['No Finding', 'Enlarged Cardiomediastinum', 'Cardiomegaly', 'Lung Opacity', 'Lung Lesion', 'Edema', 'Consolidation', 'Pneumonia', 'Atelectasis', 'Pneumothorax', 'Pleural Effusion', 'Pleural Other', 'Fracture', 'Support Devices']
labels_chex = chex[label_cols].fillna(0).astype(int)

# Check total labels from dataset
print("Samples:", len(chex))
print(labels_chex.sum().sort_values(ascending=False))

Samples: 223414
Lung Opacity                  151501
Enlarged Cardiomediastinum    143970
Support Devices               135960
Atelectasis                   123924
Cardiomegaly                  121921
Consolidation                 107229
Pleural Effusion              103650
Edema                          94370
Fracture                       61690
Pneumonia                      52226
No Finding                     35526
Lung Lesion                    31027
Pleural Other                  26325
Pneumothorax                   25339
dtype: int64


In [14]:
# Split dataset into train/val/test with patient-wise splitting
random_state = 6033689
test_ratio = 0.2
val_ratio_of_holdout = 0.5

patient_col = "patientID"
label_cols = list(labels_chex.columns) if hasattr(labels_chex, "columns") else list(range(y.shape[1]))

# Create Patient Specific Dataframe
df = chex.copy()
df[label_cols] = df[label_cols].astype(int)
patient_grouped = df.groupby(patient_col)[label_cols].max()
patient_ids = patient_grouped.index.to_numpy()
patient_labels = patient_grouped.values.astype(int) 

# First split patients (80/20) to train:temp (selection & test)
msss = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=test_ratio, random_state=random_state)
train_pat_idx, temp_pat_idx = next(msss.split(patient_ids, patient_labels))

train_patient_ids = patient_ids[train_pat_idx]
temp_patient_ids = patient_ids[temp_pat_idx]
temp_patient_labels = patient_labels[temp_pat_idx]

# Second split patients (50/50) to selection & test
msss2 = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=val_ratio_of_holdout, random_state=random_state)
selection_rel_idx, test_rel_idx = next(msss2.split(temp_patient_ids, temp_patient_labels))

selection_patient_ids = temp_patient_ids[selection_rel_idx]
test_patient_ids  = temp_patient_ids[test_rel_idx]

train_df = chex[chex[patient_col].isin(train_patient_ids)].copy()
selection_df = chex[chex[patient_col].isin(selection_patient_ids)].copy()
test_df = chex[chex[patient_col].isin(test_patient_ids)].copy()

print("Patient counts -> total/patients:", len(patient_ids))
print("Sizes -> train/val/test (patients):", len(train_patient_ids), len(selection_patient_ids), len(test_patient_ids))
print("Sizes -> train/val/test (rows):", len(train_df), len(selection_df), len(test_df))

Patient counts -> total/patients: 64540
Sizes -> train/val/test (patients): 51632 6454 6454
Sizes -> train/val/test (rows): 178447 22311 22656


In [15]:
# Making sure no patient shown twice
assert set(train_patient_ids).isdisjoint(set(test_patient_ids)), "Train/Test overlap!"
assert set(train_patient_ids).isdisjoint(set(selection_patient_ids)), "Train/Selection overlap!"
assert set(test_patient_ids).isdisjoint(set(selection_patient_ids)), "Test/Selection overlap!"

print("No patient ID overlap across train/val/test splits")

No patient ID overlap across train/val/test splits


In [16]:
full_pct = class_percentages(labels_chex.values)
train_pct = class_percentages(train_df[label_cols].values)
selection_pct = class_percentages(selection_df[label_cols].values)
test_pct = class_percentages(test_df[label_cols].values)

report_pct = pd.DataFrame({
    "full_pct": full_pct,
    "train_pct": train_pct,
    "selection_pct": selection_pct,
    "test_pct": test_pct,
})

report_pct = report_pct.sort_values("full_pct", ascending=False)
report_pct

,full_pct,train_pct,selection_pct,test_pct
Lung Opacity,67.811775,67.749528,67.858904,68.255650
Enlarged Cardiomediastinum,64.440903,64.358605,64.672135,64.861405
Support Devices,60.855631,60.862889,61.243333,60.416667
Atelectasis,55.468323,55.420377,55.878266,55.442267
Cardiomegaly,54.571782,54.490129,55.156649,54.638948
Consolidation,47.995649,47.930758,48.308010,48.199153
Pleural Effusion,46.393691,46.305065,46.887186,46.605756
Edema,42.239967,42.155374,42.284075,42.862818
Fracture,27.612415,27.628932,27.183900,27.904308
Pneumonia,23.376333,23.315046,23.580297,23.658192


In [17]:
full_count = class_counts(labels_chex.values, label_cols)
train_count = class_counts(train_df[label_cols].values, label_cols)
selection_count = class_counts(selection_df[label_cols].values, label_cols)
test_count = class_counts(test_df[label_cols].values, label_cols)

report_count = pd.DataFrame({
    "full_count": full_count,
    "train_count": train_count,
    "selection_count": selection_count,
    "test_count": test_count,
})

report_count = report_count.sort_values("full_count", ascending=False)
report_count

,full_count,train_count,selection_count,test_count
Lung Opacity,151501,120897.0,15140.0,15464.0
Enlarged Cardiomediastinum,143970,114846.0,14429.0,14695.0
Support Devices,135960,108608.0,13664.0,13688.0
Atelectasis,123924,98896.0,12467.0,12561.0
Cardiomegaly,121921,97236.0,12306.0,12379.0
Consolidation,107229,85531.0,10778.0,10920.0
Pleural Effusion,103650,82630.0,10461.0,10559.0
Edema,94370,75225.0,9434.0,9711.0
Fracture,61690,49303.0,6065.0,6322.0
Pneumonia,52226,41605.0,5261.0,5360.0


In [18]:
# Saving the split in CSV for TorchXRayVision data loader
train_df.to_csv("chex_train_split.csv", index=False)
selection_df.to_csv("train_chex_selection_split.csv", index=False)
test_df.to_csv("train_chex_test_split.csv", index=False)

## Object CXR

In [19]:
ox = pd.read_csv(r"/home/varrel/imaging/all_images/OX_dataset/train.csv")
ox['label'] = np.where(ox['annotation'].isna(), 0, 1)

In [21]:
# Split dataset into train/selection/test (no patient-wise splitting info available)
X = ox['image_name']
y = ox['label']

# First split: train (90%) selection (10%)
X_train, X_selection, y_train, y_selection = train_test_split(
    X, y,
    test_size=0.1,
    stratify=y,
    random_state=42
)

train_df = ox.loc[X_train.index]
selection_df = ox.loc[X_selection.index]

In [22]:
# Print sizes and distributions
print("Sizes -> train/selection:", len(train_df), len(selection_df))
print("Train distribution:\n", train_df['label'].value_counts(normalize=True))
print("Selection distribution:\n", selection_df['label'].value_counts(normalize=True))

Sizes -> train/selection: 7200 800
Train distribution:
 label
1    0.5
0    0.5
Name: proportion, dtype: float64
Selection distribution:
 label
0    0.5
1    0.5
Name: proportion, dtype: float64


In [23]:
# Saving the split in CSV for TorchXRayVision data loader
train_df.to_csv("ox_train_split.csv", index=False)
selection_df.to_csv("ox_selection_split.csv", index=False)

## MIMIC-CXR

In [31]:
csvpath=r"/home/varrel/imaging/physionet.org/files/mimic-cxr-jpg/2.0.0/mimic-cxr-2.0.0-chexpert.csv"
metacsvpath=r"/home/varrel/imaging/physionet.org/files/mimic-cxr-jpg/2.0.0/mimic-cxr-2.0.0-metadata.csv"
splitpath=r"/home/varrel/imaging/physionet.org/files/mimic-cxr-jpg/2.0.0/mimic-cxr-2.0.0-split.csv"

In [ ]:
split = pd.read_csv(splitpath)
csv = pd.read_csv(csvpath)
csv = csv.fillna(0) # Using U-Zeroes

In [34]:
# Create separate DataFrames
def_train_df = split[split["split"] == "train"].copy()
def_selection_df = split[split["split"] == "validate"].copy()
def_test_df = split[split["split"] == "test"].copy()

# Quick counts for MIMIC official split
print("Train size:", len(def_train_df))
print("Selection size:", len(def_selection_df))
print("Test size:", len(def_test_df))

Train size: 368960
Selection size: 2991
Test size: 5159


In [35]:
# Filter out the main CSV with the 3 datasets
exclude_study_id = pd.concat([def_selection_df["study_id"], def_test_df["study_id"]])
csv_filtered = csv[~csv["study_id"].isin(exclude_study_id)].copy()

# Using Binary Mapping (U-Zeroes) for MIMIC to reduce complexity in the model
csv_filtered = csv_filtered.replace(-1, 0)

In [36]:
# Create label DataFrame & Print summary statistics
label_cols = ['No Finding', 'Enlarged Cardiomediastinum', 'Cardiomegaly', 'Lung Opacity', 'Lung Lesion', 'Edema', 'Consolidation', 'Pneumonia', 'Atelectasis', 'Pneumothorax', 'Pleural Effusion', 'Pleural Other', 'Fracture', 'Support Devices']
labels_mimic = csv_filtered[label_cols].fillna(0).astype(int)

# Check total labels from dataset
print("Samples:", len(csv_filtered))
print(labels_mimic.sum().sort_values(ascending=False))

Samples: 222750
No Finding                    74305
Support Devices               64868
Pleural Effusion              52759
Lung Opacity                  50099
Atelectasis                   44718
Cardiomegaly                  43602
Edema                         26093
Pneumonia                     16093
Consolidation                 10476
Pneumothorax                  10171
Enlarged Cardiomediastinum     6968
Lung Lesion                    6091
Fracture                       4283
Pleural Other                  1933
dtype: int64


In [37]:
# Split dataset into train/val/test with patient-wise splitting
random_state = 6033689
test_ratio = 0.2
val_ratio_of_holdout = 0.5

patient_col = "subject_id"
label_cols = list(labels_mimic.columns) if hasattr(labels_mimic, "columns") else list(range(y.shape[1]))

# Create Patient Specific Dataframe
df = csv_filtered.copy()
df[label_cols] = df[label_cols].astype(int)
patient_grouped = df.groupby(patient_col)[label_cols].max()
patient_ids = patient_grouped.index.to_numpy()
patient_labels = patient_grouped.values.astype(int) 

# First split patients (80/20) to train:temp (selection & test)
msss = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=test_ratio, random_state=random_state)
train_pat_idx, temp_pat_idx = next(msss.split(patient_ids, patient_labels))

train_patient_ids = patient_ids[train_pat_idx]
temp_patient_ids = patient_ids[temp_pat_idx]
temp_patient_labels = patient_labels[temp_pat_idx]

# Second split patients (50/50) to selection & test
msss2 = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=val_ratio_of_holdout, random_state=random_state)
selection_rel_idx, test_rel_idx = next(msss2.split(temp_patient_ids, temp_patient_labels))

selection_patient_ids = temp_patient_ids[selection_rel_idx]
test_patient_ids  = temp_patient_ids[test_rel_idx]

train_df = csv_filtered[csv_filtered[patient_col].isin(train_patient_ids)].copy()
selection_df = csv_filtered[csv_filtered[patient_col].isin(selection_patient_ids)].copy()
test_df = csv_filtered[csv_filtered[patient_col].isin(test_patient_ids)].copy()

print("Patient counts -> total/patients:", len(patient_ids))
print("Sizes -> train/val/test (patients):", len(train_patient_ids), len(selection_patient_ids), len(test_patient_ids))
print("Sizes -> train/val/test (rows):", len(train_df), len(selection_df), len(test_df))

Patient counts -> total/patients: 64586
Sizes -> train/val/test (patients): 51668 6459 6459
Sizes -> train/val/test (rows): 178413 22009 22328


In [38]:
# Making sure no patient shown twice
assert set(train_patient_ids).isdisjoint(set(test_patient_ids)), "Train/Test overlap!"
assert set(train_patient_ids).isdisjoint(set(selection_patient_ids)), "Train/Selection overlap!"
assert set(test_patient_ids).isdisjoint(set(selection_patient_ids)), "Test/Selection overlap!"

print("No patient ID overlap across train/val/test splits")

No patient ID overlap across train/val/test splits


In [39]:
full_pct = class_percentages(labels_mimic.values)
train_pct = class_percentages(train_df[label_cols].values)
selection_pct = class_percentages(selection_df[label_cols].values)
test_pct = class_percentages(test_df[label_cols].values)

report_pct = pd.DataFrame({
    "full_pct": full_pct,
    "train_pct": train_pct,
    "selection_pct": selection_pct,
    "test_pct": test_pct,
})

report_pct = report_pct.sort_values("full_pct", ascending=False)
report_pct

,full_pct,train_pct,selection_pct,test_pct
No Finding,33.358025,33.428057,33.331819,32.824257
Support Devices,29.121437,29.228812,28.265709,29.106951
Pleural Effusion,23.685297,23.664755,23.708483,23.826585
Lung Opacity,22.491134,22.487711,22.140942,22.863669
Atelectasis,20.075421,20.106719,20.132673,19.768900
Cardiomegaly,19.574411,19.616284,19.069472,19.737549
Edema,11.714029,11.837702,11.068199,11.362415
Pneumonia,7.224691,7.215281,7.028943,7.492834
Consolidation,4.703030,4.722750,4.552683,4.693658
Pneumothorax,4.566105,4.565250,4.884365,4.259226


In [40]:
full_count = class_counts(labels_mimic.values, label_cols)
train_count = class_counts(train_df[label_cols].values, label_cols)
selection_count = class_counts(selection_df[label_cols].values, label_cols)
test_count = class_counts(test_df[label_cols].values, label_cols)

report_count = pd.DataFrame({
    "full_count": full_count,
    "train_count": train_count,
    "selection_count": selection_count,
    "test_count": test_count,
})

report_count = report_count.sort_values("full_count", ascending=False)
report_count

,full_count,train_count,selection_count,test_count
No Finding,74305,59640.0,7336.0,7329.0
Support Devices,64868,52148.0,6221.0,6499.0
Pleural Effusion,52759,42221.0,5218.0,5320.0
Lung Opacity,50099,40121.0,4873.0,5105.0
Atelectasis,44718,35873.0,4431.0,4414.0
Cardiomegaly,43602,34998.0,4197.0,4407.0
Edema,26093,21120.0,2436.0,2537.0
Pneumonia,16093,12873.0,1547.0,1673.0
Consolidation,10476,8426.0,1002.0,1048.0
Pneumothorax,10171,8145.0,1075.0,951.0


In [41]:
# Include back the MIMIC official split
merged_df = csv.merge(split[["study_id", "split"]], on="study_id", how="inner")
def_selection_df = merged_df[merged_df["split"] == "validate"].copy()
def_test_df = merged_df[merged_df["split"] == "test"].copy()

# Adding the excluded patient from before
combined_selection_df = pd.concat([def_selection_df, selection_df], ignore_index=True)
combined_test_df = pd.concat([def_test_df, test_df], ignore_index=True)

In [42]:
# Checking again the overlap
train_patient_ids = set(train_df["subject_id"])
selection_patient_ids = set(combined_selection_df["subject_id"])
test_patient_ids = set(combined_test_df["subject_id"])

assert train_patient_ids.isdisjoint(test_patient_ids), "Train/Test overlap!"
assert train_patient_ids.isdisjoint(selection_patient_ids), "Train/Selection overlap!"
assert test_patient_ids.isdisjoint(selection_patient_ids), "Test/Selection overlap!"

print("No patient ID overlap across train/val/test splits")

No patient ID overlap across train/val/test splits


In [43]:
# Saving the split in CSV for TorchXRayVision data loader
train_df.to_csv("mimic_train_split.csv", index=False)
combined_selection_df.to_csv("mimic_selection_split.csv", index=False)
combined_test_df.to_csv("mimic_test_split.csv", index=False)

## VinDr CXR

In [45]:
def majority_vote(df, id_col: str = 'image_id'):
    # Handling different radiologist vote for VinDr-CXR
    original_cols = [col for col in df.columns if col != id_col]
    summed = df.groupby(id_col)[original_cols].sum()
    group_sizes = df.groupby(id_col).size()
    majority = (summed.ge(group_sizes // 2 + 1, axis=0)).astype(int)
    majority = majority[original_cols]
    return majority.reset_index()

In [46]:
csv_train_dir = r"vindr_image_labels_train.csv"
train_sheet = pd.read_csv(csv_train_dir)
train_sheet = train_sheet.drop(columns=["rad_id"])
majority_vote_df = majority_vote(train_sheet, 'image_id')

In [48]:
# Create label DataFrame & Print summary statistics
label_cols = majority_vote_df.columns.drop(["image_id"])
labels_vin = majority_vote_df[label_cols].fillna(0).astype(int)

# Check total labels from dataset
print("Samples:", len(majority_vote_df))
print(labels_vin.sum().sort_values(ascending=False))

Samples: 15000
No finding            10601
Other diseases         4003
Aortic enlargement     2346
Cardiomegaly           1817
Pulmonary fibrosis     1017
Pleural thickening      882
Pleural effusion        634
Lung Opacity            547
Tuberculosis            482
Pneumonia               471
Nodule/Mass             409
Other lesion            362
Infiltration            245
Calcification           177
ILD                     152
Lung tumor              134
Consolidation           121
Mediastinal shift        85
Atelectasis              62
Pneumothorax             58
Rib fracture             41
Enlarged PA              21
Lung cavity              20
Emphysema                14
COPD                      7
Lung cyst                 4
Clavicle fracture         1
Edema                     1
dtype: int64


In [49]:
# Split dataset into train/selection/test (no patient-wise splitting info available)
random_state = 6033689
train_test_ratio = 0.1  #(90/10)

X = majority_vote_df.index.values
y = labels_vin.values

# First split: 80/20
msss = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=train_test_ratio, random_state=random_state)
train_idx, selection_idx = next(msss.split(X, y))

# Create DataFrames for each set
train_df = majority_vote_df.iloc[train_idx].copy()
selection_df = majority_vote_df.iloc[selection_idx].copy()

print("Sizes -> train/selection/test:", len(train_df), len(selection_df))

Sizes -> train/selection/test: 13495 1505


In [50]:
full_pct = class_percentages(labels_vin.values)
train_pct = class_percentages(labels_vin.iloc[train_idx].values)
selection_pct = class_percentages(labels_vin.iloc[selection_idx].values)

report_pct = pd.DataFrame({
    "full_pct": full_pct,
    "train_pct": train_pct,
    "selection_pct": selection_pct,
})

report_pct = report_pct.sort_values("full_pct", ascending=False)
report_pct

,full_pct,train_pct,selection_pct
No finding,70.673333,70.700259,70.431894
Other diseases,26.686667,26.698777,26.578073
Aortic enlargement,15.640000,15.642831,15.614618
Cardiomegaly,12.113333,12.115598,12.093023
Pulmonary fibrosis,6.780000,6.780289,6.777409
Pleural thickening,5.880000,5.883661,5.847176
Pleural effusion,4.226667,4.231197,4.186047
Lung Opacity,3.646667,3.645795,3.654485
Tuberculosis,3.213333,3.216006,3.189369
Pneumonia,3.140000,3.141904,3.122924


In [51]:
full_count = class_counts(labels_vin.values, label_cols)
train_count = class_counts(labels_vin.iloc[train_idx].values, label_cols)
selection_count = class_counts(labels_vin.iloc[selection_idx].values, label_cols)

report_count = pd.DataFrame({
    "full_count": full_count,
    "train_count": train_count,
    "selection_count": selection_count,
})

report_count = report_count.sort_values("full_count", ascending=False)
report_count

,full_count,train_count,selection_count
No finding,10601,9541,1060
Other diseases,4003,3603,400
Aortic enlargement,2346,2111,235
Cardiomegaly,1817,1635,182
Pulmonary fibrosis,1017,915,102
Pleural thickening,882,794,88
Pleural effusion,634,571,63
Lung Opacity,547,492,55
Tuberculosis,482,434,48
Pneumonia,471,424,47


In [53]:
# Saving the split in CSV for TorchXRayVision data loader
vin_csv = r"/home/varrel/imaging/all_images/VinDr_dataset/annotations_train.csv"
vin_annotation = pd.read_csv(vin_csv)
train_vin = vin_annotation[~vin_annotation["image_id"].isin(selection_df["image_id"])].copy()
test_vin = vin_annotation[vin_annotation["image_id"].isin(selection_df["image_id"])].copy()

train_vin.to_csv("vin_train_split.csv", index=False)
test_vin.to_csv("vin_selection_split.csv", index=False)